# Download ORena FOCUS PROCEDURE-track videos from Hugging Face

Targets the two datasets behind the PROCEDURE track:
- `orena-dkfz/heico-focus-vqa` (HeiCo-FOCUS, 30 colorectal videos, public)
- `orena-dkfz/lapchole-focus-vqa` (LapChole-FOCUS, 170 lap chole videos, gated --
  request-only, this is almost certainly what your token is for)

**Structural unknown, resolved by Stage 1 below rather than assumed:** HF
datasets store video either as loose files in the repo, or as bytes embedded
inside parquet columns. This notebook checks which one applies before
downloading anything, and branches accordingly.

## 0. Setup

In [ ]:

import sys, subprocess, getpass, re
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/endofm-benchmark")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "huggingface_hub", "datasets", "pyarrow"], check=True)
else:
    PROJECT_ROOT = Path(r"E:/endofm-benchmark")

DATA_DIR = PROJECT_ROOT / "data" / "orena-procedure"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Data dir:", DATA_DIR)


## 1. Authenticate

In [ ]:

from huggingface_hub import login

HF_TOKEN = getpass.getpass("Hugging Face access token: ")
login(token=HF_TOKEN)
print("Logged in (token not displayed or saved to disk).")


## 2. Stage 1 -- discover each repo's actual file structure

Lists every file in each dataset repo and classifies it: loose video file,
parquet (may embed video bytes), or other. Read this output before Stage 2
-- it decides which download path actually applies.

In [ ]:

from huggingface_hub import list_repo_files
from huggingface_hub.utils import GatedRepoError, RepositoryNotFoundError

REPO_IDS = [
    "orena-dkfz/heico-focus-vqa",
    "orena-dkfz/lapchole-focus-vqa",
]

VIDEO_EXTS = (".mp4", ".avi", ".mov", ".mkv")

repo_files = {}
for repo_id in REPO_IDS:
    try:
        files = list_repo_files(repo_id, repo_type="dataset", token=HF_TOKEN)
    except GatedRepoError:
        print(f"{repo_id}: GATED and not yet approved for this token -- request access on the HF page first.")
        continue
    except RepositoryNotFoundError:
        print(f"{repo_id}: not found (wrong repo id, or no access).")
        continue

    video_files = [f for f in files if f.lower().endswith(VIDEO_EXTS)]
    parquet_files = [f for f in files if f.lower().endswith(".parquet")]
    other_files = [f for f in files if f not in video_files and f not in parquet_files]

    repo_files[repo_id] = {"video": video_files, "parquet": parquet_files, "other": other_files}
    print(f"\n{repo_id}: {len(files)} total files")
    print(f"  loose video files: {len(video_files)}")
    print(f"  parquet files:     {len(parquet_files)}  {parquet_files[:5]}")
    print(f"  other files:       {len(other_files)}  {other_files[:5]}")


## 3. Stage 2a -- download loose video files (if Stage 1 found any)

Uses `hf_hub_download`, which handles resume/caching internally -- no need
to hand-roll resumable-download logic like the raw-HTTP notebooks did.

In [ ]:

from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

MAX_VIDEOS = 5  # total across both repos, shared with Stage 2b below

# Pre-existing heico videos on Drive -- skip the download if already there.
HEICO_VIDEOS_DIR = (Path("/content/drive/MyDrive/Debjyoti/orena-procedure/heico-focus-vqa/videos")
                     if IN_COLAB else None)

downloaded_video_paths = []
heico_satisfied = False

if HEICO_VIDEOS_DIR is not None and HEICO_VIDEOS_DIR.is_dir():
    existing = sorted(p for p in HEICO_VIDEOS_DIR.iterdir() if p.suffix.lower() in VIDEO_EXTS)
    if len(existing) >= MAX_VIDEOS:
        downloaded_video_paths = existing[:MAX_VIDEOS]
        heico_satisfied = True
        print(f"Found {len(existing)} video(s) already in {HEICO_VIDEOS_DIR} -- skipping heico download.")

for repo_id, info in repo_files.items():
    if heico_satisfied and repo_id.endswith("heico-focus-vqa"):
        continue
    if not info["video"] or len(downloaded_video_paths) >= MAX_VIDEOS:
        continue
    out_dir = DATA_DIR / repo_id.split("/")[-1] / "videos"
    out_dir.mkdir(parents=True, exist_ok=True)
    remaining = MAX_VIDEOS - len(downloaded_video_paths)
    for fname in tqdm(info["video"][:remaining], desc=f"{repo_id} videos"):
        local_path = hf_hub_download(repo_id=repo_id, filename=fname, repo_type="dataset",
                                      token=HF_TOKEN, local_dir=out_dir)
        downloaded_video_paths.append(Path(local_path))

print(f"Downloaded {len(downloaded_video_paths)} loose video file(s).")


## 4. Stage 2b -- extract video bytes from parquet (if Stage 1 found no loose videos)

Only runs anything if Stage 1 found parquet files and no loose videos.
Loads each parquet, looks for a column that plausibly holds video bytes
(binary data, or a dict with a `bytes`/`path` structure -- the standard
shape for an HF `datasets.Video`/`Image`-like feature), and writes each
row's video out as its own file.

In [ ]:

import pandas as pd

def find_video_like_columns(df):
    candidates = []
    for col in df.columns:
        sample = df[col].dropna().iloc[0] if df[col].notna().any() else None
        if isinstance(sample, bytes):
            candidates.append((col, "bytes"))
        elif isinstance(sample, dict) and ("bytes" in sample or "path" in sample):
            candidates.append((col, "dict"))
    return candidates

extracted_from_parquet = []
for repo_id, info in repo_files.items():
    if info["video"] or not info["parquet"]:
        continue  # only handle the parquet-embedded case when no loose videos exist
    if len(downloaded_video_paths) + len(extracted_from_parquet) >= MAX_VIDEOS:
        break

    out_dir = DATA_DIR / repo_id.split("/")[-1] / "videos_from_parquet"
    out_dir.mkdir(parents=True, exist_ok=True)

    for pq_name in info["parquet"]:
        local_pq = hf_hub_download(repo_id=repo_id, filename=pq_name, repo_type="dataset", token=HF_TOKEN)
        df = pd.read_parquet(local_pq)
        video_cols = find_video_like_columns(df)
        print(f"{repo_id}/{pq_name}: columns={list(df.columns)} video-like={video_cols}")

        for col, kind in video_cols:
            for i, row in df.iterrows():
                if len(downloaded_video_paths) + len(extracted_from_parquet) >= MAX_VIDEOS:
                    break
                cell = row[col]
                if cell is None:
                    continue
                if kind == "bytes":
                    data, ext = cell, ".mp4"
                else:
                    data = cell.get("bytes")
                    ext = Path(cell.get("path", "")).suffix or ".mp4"
                    if data is None:
                        continue
                out_path = out_dir / f"{Path(pq_name).stem}_{col}_{i}{ext}"
                if not out_path.exists():
                    out_path.write_bytes(data)
                extracted_from_parquet.append(out_path)

print(f"\nExtracted {len(extracted_from_parquet)} video file(s) from parquet.")
if not extracted_from_parquet and not downloaded_video_paths:
    print("Neither path found actual video data -- inspect the 'other' files list from Stage 1, "
          "the video data may be referenced by URL/path inside the parquet instead of embedded directly.")


## 5. Sanity check

In [ ]:

all_videos = downloaded_video_paths + extracted_from_parquet
total_bytes = sum(p.stat().st_size for p in all_videos if p.exists())
print(f"Total video files: {len(all_videos)}")
print(f"Total size: {total_bytes / 1e9:.1f} GB")
print("Sample paths:", [str(p) for p in all_videos[:5]])


## Notes / caveats

- **Gated repo**: `orena-dkfz/lapchole-focus-vqa` requires approved access on
  the HF dataset page -- if Stage 1 reports `GatedRepoError`, the token
  itself is fine but the access request hasn't been approved yet (or wasn't
  made) for that account.
- **Parquet-embedded video is a guess at the schema**, not a verified one --
  Stage 1's printed column list is what actually determines whether Stage 2b's
  column-detection logic applies as written, or needs adjusting to match the
  real column name/structure.
- This notebook only fetches raw video -- pairing it with the VQA
  question/answer parquet data (for training/eval) is a separate step.
